In [28]:
import requests
import yaml
import pandas as pd
from pathlib import Path

# --- Cambiá estas dos líneas por el disco que quieras probar ---
ARTIST = "Artic Monkeys"
ALBUM = "Whatever People Say I Am, That's What I'm Not"


with open(Path.cwd() / "config.local.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}
API_KEY = cfg["lastfm"]["api_key"]

API_URL = "https://ws.audioscrobbler.com/2.0/"
HEADERS = {"User-Agent": "navidrome-style-tagger/0.1 (uso personal)"}

In [29]:
def lastfm(method, **params):
    r = requests.get(
        API_URL,
        params={"method": method, "api_key": API_KEY, "format": "json", "autocorrect": 1, **params},
        headers=HEADERS,
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"Last.fm error {data['error']}: {data.get('message')}")
    return data


def top_tags(method, **params):
    data = lastfm(method, **params)
    tags = data.get("toptags", {}).get("tag", [])
    if isinstance(tags, dict):   # si hay un solo tag, a veces viene como dict
        tags = [tags]
    return pd.DataFrame(
        [{"tag": t["name"], "peso": int(t["count"])} for t in tags]
    ).sort_values("peso", ascending=False, ignore_index=True) if tags else pd.DataFrame(columns=["tag", "peso"])

In [30]:
import re

# Estilos autorizados. Editá libremente: sacá lo que no quieras y sumá lo que descubras en "Descartados".
AUTORIZADOS = [
    # tu lista
    "alternative rock", "progressive rock", "indie rock", "latin", "progressive metal",
    "punk rock", "classic rock", "classical", "children", "pop rock",
    # prog y afines
    "symphonic prog", "symphonic rock", "art rock", "neo-prog", "canterbury scene",
    "krautrock", "space rock", "post-rock", "math rock", "jazz fusion", "jazz rock",
    "guitar virtuoso",
    # rock
    "hard rock", "psychedelic rock", "folk rock", "blues rock", "southern rock",
    "glam rock", "garage rock", "post-punk", "new wave", "shoegaze", "grunge",
    "britpop", "power pop", "pop rock", "rock nacional",
    # metal
    "heavy metal", "thrash metal", "death metal", "black metal", "doom metal",
    "power metal", "folk metal", "gothic metal",
    # fuera del rock
    "jazz", "blues", "folk", "country", "singer-songwriter", "soul", "funk",
    "reggae", "ska", "hip-hop", "electronic", "ambient", "new age", "synthpop",
    "soundtrack", "tango", "bossa nova",
]

# Variantes de escritura que se suman al estilo canónico (la clave es lo que devuelve Last.fm).
ALIAS = {
    "prog rock": "progressive rock",
    "prog": "progressive rock",
    "progressive": "progressive rock",
    "prog metal": "progressive metal",
    "alt rock": "alternative rock",
    "classical music": "classical",
    "kids": "children",
    "childrens music": "children",
    "latin music": "latin",
    "neo prog": "neo-prog",
}

def norm(t):
    return re.sub(r"[\s\-_/]+", " ", t.lower()).strip()

CANON = {norm(g): g for g in AUTORIZADOS}
CANON.update({norm(a): c for a, c in ALIAS.items()})


def filtrar(df):
    """Devuelve (tabla de estilos autorizados con peso recalculado, lista de descartados)."""
    ok, descartados = {}, []
    for r in df.itertuples():
        canon = CANON.get(norm(r.tag))
        if canon:
            ok[canon] = ok.get(canon, 0) + r.peso   # alias y variantes se suman
        else:
            descartados.append((r.tag, r.peso))
    out = pd.DataFrame([{"estilo": k, "peso original": v} for k, v in ok.items()])
    if len(out):
        out["% entre autorizados"] = (100 * out["peso original"] / out["peso original"].sum()).round(1)
        out = out.sort_values("peso original", ascending=False, ignore_index=True)
    return out, descartados


def mostrar(titulo, df):
    ok, desc = filtrar(df)
    print(f"=== {titulo}: {len(ok)} estilos autorizados de {len(df)} tags ===")
    if len(ok):
        display(ok)
    else:
        print("(ninguno autorizado)")
    print("Descartados:", ", ".join(f"{t} ({p})" for t, p in desc[:20]) or "-")
    print()

In [31]:
album_tags = top_tags("album.getTopTags", artist=ARTIST, album=ALBUM)
artist_tags = top_tags("artist.getTopTags", artist=ARTIST)

mostrar(f"Álbum: {ARTIST} - {ALBUM}", album_tags)
mostrar(f"Artista: {ARTIST}", artist_tags)

=== Álbum: Artic Monkeys - Whatever People Say I Am, That's What I'm Not: 3 estilos autorizados de 10 tags ===


,estilo,peso original,% entre autorizados
0,indie rock,100,90.9
1,alternative rock,8,7.3
2,garage rock,2,1.8


Descartados: 2006 (84), rock (47), indie (43), british (19), 00s (11), Post-punk revival (2), favourite albums (2)

=== Artista: Artic Monkeys: 5 estilos autorizados de 10 tags ===


,estilo,peso original,% entre autorizados
0,indie rock,100,91.7
1,alternative rock,5,4.6
2,britpop,2,1.8
3,garage rock,1,0.9
4,post-punk,1,0.9


Descartados: indie (92), british (54), alternative (41), rock (20), Post-punk revival (1)



# --- Diagnóstico: ¿por qué no aparece? Probamos varias vías y mostramos qué responde Last.fm ---
import json

def probe(titulo, fn):
    print(f"--- {titulo} ---")
    try:
        fn()
    except Exception as e:
        print("  ERROR:", e)

def listar(x):
    return [x] if isinstance(x, dict) else (x or [])

def p_album_top():
    d = lastfm("album.getTopTags", artist=ARTIST, album=ALBUM)
    print(json.dumps(d, ensure_ascii=False)[:500])

def p_album_info():
    d = lastfm("album.getInfo", artist=ARTIST, album=ALBUM)["album"]
    print(d.get("artist"), "|", d.get("name"), "|", [t["name"] for t in listar((d.get("tags") or {}).get("tag"))])

def p_track_top():
    df = top_tags("track.getTopTags", artist=ARTIST, track=ALBUM)
    print(df.head(10).to_string(index=False) if len(df) else "  (sin tags)")

def p_album_search():
    for a in listar(lastfm("album.search", album=ALBUM, limit=5)["results"]["albummatches"]["album"]):
        print(" ", a["artist"], "|", a["name"])

def p_track_search():
    for t in listar(lastfm("track.search", track=ALBUM, limit=5)["results"]["trackmatches"]["track"]):
        print(" ", t["artist"], "|", t["name"])

probe("album.getTopTags (respuesta cruda)", p_album_top)
probe("album.getInfo (artista | nombre | tags)", p_album_info)
probe("track.getTopTags (usando ALBUM como nombre de tema)", p_track_top)
probe("album.search (cómo figura el álbum y con qué artista)", p_album_search)
probe("track.search (cómo figura el tema y con qué artista)", p_track_search)

# --- Probar todas las variantes de artista con las que Last.fm tiene este álbum ---
import time

def listar(x):
    return [x] if isinstance(x, dict) else (x or [])

res = lastfm("album.search", album=ALBUM, limit=30)["results"]["albummatches"]["album"]
variantes = sorted({a["artist"] for a in listar(res) if a["name"].lower() == ALBUM.lower()})

filas = []
for art in variantes:
    try:
        df = top_tags("album.getTopTags", artist=art, album=ALBUM, autocorrect=0)
    except Exception as e:
        print(f"{art}: ERROR {e}")
        continue
    print(f"{art}: {len(df)} tags")
    filas += [{"variante de artista": art, "tag": r.tag, "peso": r.peso} for r in df.itertuples()]
    time.sleep(0.3)

if filas:
    display(pd.DataFrame(filas))
    union = pd.DataFrame(filas).groupby("tag")["peso"].max().sort_values(ascending=False)
    print("Unión de variantes (peso máximo por tag):")
    display(union.to_frame())
else:
    print("Ninguna variante tiene tags de álbum.")